In [2]:
import sys
print(sys.executable)

C:\Users\Madhu\anaconda3\envs\deep-fake\python.exe


In [ ]:
import sys
!{sys.executable} -m pip install geopandas folium geopy shapely

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point
from geopy.distance import geodesic
import numpy as np
import folium

# ======================
# 1️⃣  PARAMETERS
# ======================
n = 1     # grid size in km
m = 2     # scan radius in km
deg_per_km = 0.009  # approx degrees per km
sigma = m / 2       # for Gaussian decay

# Chennai bounding box (approx)
min_lat, max_lat = 12.8, 13.2
min_lon, max_lon = 80.0, 80.4

# ======================
# 2️⃣  WEIGHT CONFIGURATION
# ======================
weights = {
    "schools": 3.0,
    "offices": 2.5,
    "apartments": 3.0,
    "tourist_places": 2.0,
    "bakeries": -3.0,
    "supermarkets": -2.0
}

# ======================
# 3️⃣  LOAD CSV FILES
# ======================
poi_files = {
    "schools": "chennai_schools.csv",
    "offices": "chennai_offices.csv",
    "apartments": "chennai_apartments.csv",
    "tourist_places": "chennai_tourist_places.csv",
    "bakeries": "chennai_bakeries.csv",
    "supermarkets": "chennai_supermarkets.csv"
}

pois = {}
for key, file in poi_files.items():
    df = pd.read_csv(file)
    # Access by column names 'lat' and 'lon' (no matter their position)
    df = df[['lat', 'lon']]
    df = df.rename(columns={'lat': 'Latitude', 'lon': 'Longitude'})
    pois[key] = df

# ======================
# 4️⃣  GRID CREATION
# ======================
cells = []
cell_id = 0
lat_range = np.arange(min_lat, max_lat, n * deg_per_km)
lon_range = np.arange(min_lon, max_lon, n * deg_per_km)

for lat in lat_range:
    for lon in lon_range:
        cell_id += 1
        center_lat, center_lon = lat + (n * deg_per_km / 2), lon + (n * deg_per_km / 2)
        cells.append({
            "CellID": cell_id,
            "Lat": center_lat,
            "Lon": center_lon
        })

df_cells = pd.DataFrame(cells)

# ======================
# 5️⃣  GAUSSIAN DISTANCE DECAY
# ======================
def gaussian_weight(distance_km, sigma):
    return np.exp(- (distance_km ** 2) / (2 * sigma ** 2))

# ======================
# 6️⃣  CALCULATE SCORES
# ======================
results = []

for _, cell in df_cells.iterrows():
    center = (cell.Lat, cell.Lon)
    score_components = {}
    total_raw_score = 0

    for key, df in pois.items():
        influence = 0
        for _, row in df.iterrows():
            dist = geodesic(center, (row["Latitude"], row["Longitude"])).km
            if dist <= m:
                influence += gaussian_weight(dist, sigma)
        weighted_score = influence * weights[key]
        score_components[key] = weighted_score
        total_raw_score += weighted_score

    results.append({
        "CellID": cell.CellID,
        "Lat": cell.Lat,
        "Lon": cell.Lon,
        **score_components,
        "RawScore": total_raw_score
    })

df_results = pd.DataFrame(results)

# ======================
# 7️⃣  NORMALIZE TO 0–1000
# ======================
min_s, max_s = df_results["RawScore"].min(), df_results["RawScore"].max()
df_results["Score_0_1000"] = ((df_results["RawScore"] - min_s) / (max_s - min_s)) * 1000
df_results["Rank"] = df_results["Score_0_1000"].rank(ascending=False).astype(int)

# ======================
# 8️⃣  SAVE OUTPUT
# ======================
df_results.to_csv("bakery_location_scores.csv", index=False)
print("✅ Bakery location scores saved as bakery_location_scores.csv")

# ======================
# 9️⃣  OPTIONAL MAP VISUALIZATION
# ======================
mymap = folium.Map(location=[13.0827, 80.2707], zoom_start=11)
for _, row in df_results.iterrows():
    color = "green" if row.Score_0_1000 > 700 else "orange" if row.Score_0_1000 > 300 else "red"
    folium.CircleMarker(
        location=[row.Lat, row.Lon],
        radius=5,
        color=color,
        fill=True,
        popup=(f"Cell {row.CellID}<br>"
               f"Score: {row.Score_0_1000:.1f}<br>"
               f"Raw: {row.RawScore:.2f}<br>"
               f"Rank: {row.Rank}")
    ).add_to(mymap)

mymap.save("bakery_heatmap.html")
print("🗺️ Heatmap saved as bakery_heatmap.html")

In [5]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point
from geopy.distance import geodesic
import numpy as np
import folium

# ======================
# 1️⃣  PARAMETERS (FIXED)
# ======================
n = 1     # grid size in km
m = 2     # scan radius in km

# More accurate Chennai bounding box (reduced to actual city area)
min_lat, max_lat = 12.85, 13.20   # Reduced from 12.8-13.2
min_lon, max_lon = 80.13, 80.29   # Reduced from 80.0-80.4

# ======================
# 2️⃣  ACCURATE GRID CREATION
# ======================
def create_grid(min_lat, max_lat, min_lon, max_lon, cell_size_km):
    """
    Create a precise grid with proper km-based spacing
    """
    # Calculate approximate degrees for the given km distance
    # At Chennai's latitude, 1° lat ≈ 110.574 km, 1° lon ≈ 111.320 km
    lat_deg_per_km = 1 / 110.574
    lon_deg_per_km = 1 / 111.320
    
    lat_step = cell_size_km * lat_deg_per_km
    lon_step = cell_size_km * lon_deg_per_km
    
    cells = []
    cell_id = 0
    
    # Calculate number of cells
    lat_cells = int((max_lat - min_lat) / lat_step)
    lon_cells = int((max_lon - min_lon) / lon_step)
    
    print(f"Grid dimensions: {lat_cells} × {lon_cells} = {lat_cells * lon_cells} total cells")
    
    for i in range(lat_cells):
        for j in range(lon_cells):
            cell_id += 1
            center_lat = min_lat + (i * lat_step) + (lat_step / 2)
            center_lon = min_lon + (j * lon_step) + (lon_step / 2)
            
            cells.append({
                "CellID": cell_id,
                "Lat": center_lat,
                "Lon": center_lon
            })
    
    return pd.DataFrame(cells)

# ======================
# 3️⃣  WEIGHT CONFIGURATION
# ======================
weights = {
    "schools": 3.0,
    "offices": 2.5,
    "apartments": 3.0,
    "tourist_places": 2.0,
    "bakeries": -3.0,
    "supermarkets": -2.0
}

# ======================
# 4️⃣  LOAD CSV FILES
# ======================
poi_files = {
    "schools": "chennai_schools.csv",
    "offices": "chennai_offices.csv",
    "apartments": "chennai_apartments.csv",
    "tourist_places": "chennai_tourist_places.csv",
    "bakeries": "chennai_bakeries.csv",
    "supermarkets": "chennai_supermarkets.csv"
}

pois = {}
for key, file in poi_files.items():
    try:
        df = pd.read_csv(file)
        # Access by column names 'lat' and 'lon'
        df = df[['lat', 'lon']]
        df = df.rename(columns={'lat': 'Latitude', 'lon': 'Longitude'})
        pois[key] = df
        print(f"✅ Loaded {len(df)} {key}")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")
        # Create empty dataframe if file doesn't exist
        pois[key] = pd.DataFrame(columns=['Latitude', 'Longitude'])

# ======================
# 5️⃣  CREATE GRID
# ======================
df_cells = create_grid(min_lat, max_lat, min_lon, max_lon, n)
print(f"✅ Created grid with {len(df_cells)} cells")

# ======================
# 6️⃣  GAUSSIAN DISTANCE DECAY
# ======================
def gaussian_weight(distance_km, sigma):
    return np.exp(- (distance_km ** 2) / (2 * sigma ** 2))

# ======================
# 7️⃣  CALCULATE SCORES (OPTIMIZED)
# ======================
sigma = m / 2
results = []

print("Calculating scores for each cell...")
for idx, cell in df_cells.iterrows():
    if idx % 50 == 0:  # Progress indicator
        print(f"Processing cell {idx+1}/{len(df_cells)}")
    
    center = (cell.Lat, cell.Lon)
    score_components = {}
    total_raw_score = 0

    for key, df in pois.items():
        influence = 0
        for _, row in df.iterrows():
            try:
                dist = geodesic(center, (row["Latitude"], row["Longitude"])).km
                if dist <= m:
                    influence += gaussian_weight(dist, sigma)
            except:
                continue  # Skip invalid coordinates
        
        weighted_score = influence * weights[key]
        score_components[key] = weighted_score
        total_raw_score += weighted_score

    results.append({
        "CellID": cell.CellID,
        "Lat": cell.Lat,
        "Lon": cell.Lon,
        **score_components,
        "RawScore": total_raw_score
    })

df_results = pd.DataFrame(results)

# ======================
# 8️⃣  NORMALIZE TO 0–1000
# ======================
min_s, max_s = df_results["RawScore"].min(), df_results["RawScore"].max()
if max_s > min_s:  # Avoid division by zero
    df_results["Score_0_1000"] = ((df_results["RawScore"] - min_s) / (max_s - min_s)) * 1000
else:
    df_results["Score_0_1000"] = 500  # Default middle value if all scores same

df_results["Rank"] = df_results["Score_0_1000"].rank(ascending=False).astype(int)

# ======================
# 9️⃣  SAVE OUTPUT
# ======================
df_results.to_csv("bakery_location_scores_fixed.csv", index=False)
print(f"✅ Bakery location scores saved with {len(df_results)} cells")

# Show top 10 locations
print("\n🏆 TOP 10 LOCATIONS:")
top_10 = df_results.nlargest(10, 'Score_0_1000')[['CellID', 'Lat', 'Lon', 'Score_0_1000', 'Rank']]
print(top_10.to_string(index=False))

✅ Loaded 209 schools
✅ Loaded 410 offices
✅ Loaded 3087 apartments
✅ Loaded 1109 tourist_places
✅ Loaded 105 bakeries
✅ Loaded 218 supermarkets
Grid dimensions: 38 × 17 = 646 total cells
✅ Created grid with 646 cells
Calculating scores for each cell...
Processing cell 1/646
Processing cell 51/646
Processing cell 101/646
Processing cell 151/646
Processing cell 201/646
Processing cell 251/646
Processing cell 301/646
Processing cell 351/646
Processing cell 401/646
Processing cell 451/646
Processing cell 501/646
Processing cell 551/646
Processing cell 601/646
✅ Bakery location scores saved with 646 cells

🏆 TOP 10 LOCATIONS:
 CellID       Lat       Lon  Score_0_1000  Rank
  270.0 12.990178 80.260255   1000.000000     1
   60.0 12.881653 80.206356    994.126624     2
  317.0 13.017309 80.224323    924.547762     3
  253.0 12.981134 80.260255    884.660839     4
  334.0 13.026352 80.224323    873.621245     5
   77.0 12.890697 80.206356    835.686951     6
  316.0 13.017309 80.215340    831.

In [6]:
#🔟  MAP VISUALIZATION

mymap = folium.Map(location=[13.0827, 80.2707], zoom_start=11)

# Color coding based on percentiles
for _, row in df_results.iterrows():
    if row.Score_0_1000 > df_results['Score_0_1000'].quantile(0.9):
        color = "green"  # Top 20%
    elif row.Score_0_1000 > df_results['Score_0_1000'].quantile(0.5):
        color = "orange"  # Top 50%
    else:
        color = "red"     # Bottom 50%
    
    folium.CircleMarker(
        location=[row.Lat, row.Lon],
        radius=5,
        color=color,
        fill=True,
        popup=(f"Cell {row.CellID}<br>"
               f"Score: {row.Score_0_1000:.1f}<br>"
               f"Rank: {row.Rank}")
    ).add_to(mymap)

mymap.save("bakery_heatmap_fixed.html")
print("🗺️ Heatmap saved as bakery_heatmap_fixed.html")

🗺️ Heatmap saved as bakery_heatmap_fixed.html
